# Trace GPT-2 **decode** — turn an inference into a measurement

### What this is for

`dynshape` derives kernel shapes for any `(batch, seq_len, mode)` from three traced anchors. For
**prefill** that is verified: it reproduces all 25 shipped GPT-2 templates exactly.

For **decode** it is not verified, because **no decode workload ships with the EnergAIzer artifact** —
all 90 files are `modeprefill`, even though the authors' own `run_gpt2.sh` has `MODE="prefill decode"`.
So decode currently rests on two unchecked assumptions:

1. decode runs the **same 242 kernels in the same order** as prefill
2. the sequence exponent splits into query and key axes by a **hand-written rule**

This notebook replaces both with measurements. It traces GPT-2 in decode mode, and answers:

| # | question | how |
|---|---|---|
| 1 | which kernels actually run, and in what order? | read the trace |
| 2 | are the inferred shapes right? | compare inferred vs measured, entry by entry |
| 3 | does a decode law generalise? | learn from 3 anchors, test on a **4th held-out** shape |

### You do not need an A100

Tracing records **which ops ran and their tensor shapes** — not timings. Any GPU works, and so does
CPU. A free Colab **T4** is the safest choice (the artifact is pinned to `torch==2.7.1`, which is
happiest on CUDA); CPU works but is slower and exercises less-tested bf16 paths.

The A100 is only needed by EnergAIzer's LUT, which already exists.

## 1 — Clone the artifact and install its pinned dependencies

In [ ]:
import os, subprocess, sys

EN_ROOT = "/content/single_kernel_GPU_model"
PKG     = os.path.join(EN_ROOT, "energaizer-ispass26-artifact-main")
CODE    = os.path.join(PKG, "test", "code")

if not os.path.isdir(PKG):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/shubhamOjha1000/single_kernel_GPU_model.git",
                    EN_ROOT], check=True)

# Versions from the artifact's own misc/conda_env.yml -- do not drift from these.
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "torch==2.7.1", "transformers==4.51.3", "torchlens",
                "numpy", "pandas"], check=True)

sys.path.insert(0, CODE)
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 2 — The workload config

`run_model.py` needs a config JSON, and `workload_config/` **does not ship** with the artifact. We
reconstruct it from the shipped templates themselves:

| evidence in `gpt2model_gpt2_pbf16_b8_s128_modeprefill.json` | implies |
|---|---|
| `layernorm dim = 768` | `n_embd = 768` |
| `attn batch = 96` at b=8 | `n_head = 12` |
| the block repeats 12 times | `n_layer = 12` |
| QKV `dimN = 2304` = 3 × 768 | standard fused QKV |
| MLP `dimN = 3072` = 4 × 768 | standard |

That is GPT-2 base. The one value we must override is `n_positions`: the default is **1024**, but
templates exist at `s4096`, so the authors clearly raised it too.

In [ ]:
import json, os

CONFIG_DIR = os.path.join(CODE, "workload_config", "gpt2")
os.makedirs(CONFIG_DIR, exist_ok=True)
CONFIG_FILE = os.path.join(CONFIG_DIR, "gpt2.json")   # stem must be "gpt2" -> filenames match

with open(CONFIG_FILE, "w") as f:
    json.dump({"n_embd": 768, "n_head": 12, "n_layer": 12,
               "vocab_size": 50257, "n_positions": 8192}, f, indent=1)

print(open(CONFIG_FILE).read())

## 3 — The tracing helper

We call the artifact's own `get_model` / `get_input` / `run_torchlens` **directly** rather than going
through `run_model.py`'s CLI, for two reasons:

- its `--trace` path calls `get_time_per_iter()` first, which calls `torch.cuda.synchronize()` — that
  fails outright on CPU;
- the whole body sits inside a bare `except Exception as e: print(e)`, so a failure prints a line and
  silently produces no trace.

Same tracing function, no swallowed errors. We round-trip through CSV exactly as the real pipeline
does, so `parse_trace` sees identical input.

In [ ]:
import pandas as pd
from run_model import get_model, get_input, run_torchlens
from parse_trace import parse

MODEL_TYPE = ("LanguageModel", "GPT2Model")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.bfloat16
OUT    = "/content/traces"; os.makedirs(OUT, exist_ok=True)

def trace_shape(batch, seqlen, mode, prec="bf16"):
    """Trace one (batch, seqlen, mode) and return (entries, csv_path, json_path)."""
    name = f"gpt2model_gpt2_p{prec}_b{batch}_s{seqlen}_mode{mode}"
    csv_path  = os.path.join(OUT, name + ".csv")
    json_path = os.path.join(OUT, name + ".json")

    module, config = get_model(MODEL_TYPE, CONFIG_FILE, DEVICE, DTYPE, "eager")
    inp = get_input(MODEL_TYPE, config, batch, seqlen, mode, DTYPE, DEVICE)

    try:
        df = run_torchlens(module, MODEL_TYPE, inp["input"], inp["past_kv"],
                           inp["use_cache"], seqlen)
    except Exception as e:
        # transformers 4.51 may refuse the legacy list-of-[k,v] cache that
        # get_input() builds. Converting is a compatibility shim, not a no-op:
        # note it, because it could change which branch the model takes.
        if mode != "decode":
            raise
        print(f"  [shim] legacy cache rejected ({type(e).__name__}); "
              f"retrying with DynamicCache")
        from transformers.cache_utils import DynamicCache
        cache = DynamicCache.from_legacy_cache(
            tuple((kv[0], kv[1]) for kv in inp["past_kv"]))
        df = run_torchlens(module, MODEL_TYPE, inp["input"], cache,
                           inp["use_cache"], seqlen)

    df.to_csv(csv_path, index=False)
    entries = parse(pd.read_csv(csv_path), False)
    with open(json_path, "w") as f:
        json.dump(entries, f)

    del module; import gc; gc.collect()
    if DEVICE == "cuda": torch.cuda.empty_cache()
    print(f"  {name}: {len(entries)} kernels")
    return entries, csv_path, json_path


def norm(entries):
    """Canonical form for comparison.

    `parse()` returns (dict, tuple); `json.load()` returns [dict, list]. A raw
    `==` between them is always False, which would make the control below fail
    for a reason that has nothing to do with the trace.
    """
    return [(dict(q), tuple(op)) for q, op in entries]


print("device:", DEVICE)

## 4 — The control: reproduce a template that already exists

**Do this before touching decode.** Trace `(b8, s128, prefill)` and compare against the shipped
`gpt2model_gpt2_pbf16_b8_s128_modeprefill.json`.

If our pipeline reproduces a known file **exactly**, then decode traces from the same pipeline can be
trusted. If it does not — wrong config, wrong transformers version, wrong attention backend — we find
out here, on a case with a known answer, instead of misreading a decode result later.

In [ ]:
SHIPPED = os.path.join(PKG, "test", "data", "workloads", "all")

ctrl, _, _ = trace_shape(8, 128, "prefill")
ctrl = norm(ctrl)
ref = norm(json.load(open(os.path.join(SHIPPED,
           "gpt2model_gpt2_pbf16_b8_s128_modeprefill.json"))))

print(f"\n  traced : {len(ctrl)} kernels")
print(f"  shipped: {len(ref)} kernels")

CONTROL_OK = (ctrl == ref)
if CONTROL_OK:
    print("\n  EXACT MATCH -- the pipeline reproduces a known template.")
else:
    print("\n  MISMATCH -- investigate before trusting any decode result below.")
    if len(ctrl) == len(ref):
        for i, (a, b) in enumerate(zip(ctrl, ref)):
            if a != b:
                print(f"    first difference at entry {i}:")
                print(f"      traced : {a}")
                print(f"      shipped: {b}")
                break
    else:
        from collections import Counter
        print("    traced op mix :", Counter(tuple(e[1]) for e in ctrl))
        print("    shipped op mix:", Counter(tuple(e[1]) for e in ref))

## 5 — Trace decode

Four shapes. Three are the anchors that teach the scaling law; the fourth is **held out** so the
learned law can be tested against a trace it has never seen — the same proof structure that gives
prefill its 25/25.

| shape | role |
|---|---|
| `b8, s128` | base |
| `b16, s128` | batch doubled → recovers the batch exponent |
| `b8, s512` | context ×4 → recovers the context exponent |
| `b16, s512` | **held out** — never used for learning, only for testing |

In decode, `seqlen` is the **KV cache length**: one new token per sequence attends over that many
stored tokens.

In [ ]:
decode_traces = {}
for b, s in [(8, 128), (16, 128), (8, 512), (16, 512)]:
    entries, _, _ = trace_shape(b, s, "decode")
    decode_traces[(b, s)] = norm(entries)

print("\ndone:", {k: len(v) for k, v in decode_traces.items()})

## 6 — Question 1: which kernels run, and in what order?

In [ ]:
from collections import Counter

dec = decode_traces[(8, 128)]
pre = ctrl

print(f"prefill : {len(pre)} kernels")
print(f"decode  : {len(dec)} kernels")
print()

same_len = len(pre) == len(dec)
same_seq = same_len and [tuple(e[1]) for e in pre] == [tuple(e[1]) for e in dec]

if same_seq:
    print("The op SEQUENCE is identical. The assumption behind the inferred rule holds:")
    print("decode runs the same kernels in the same order, only the shapes differ.")
else:
    print("The op sequence DIFFERS. The inferred rule was wrong about structure,")
    print("not just about exponents -- decode needs its own template, which is")
    print("exactly what these traces now provide.")

print(f"\n{'op':<28}{'prefill':>9}{'decode':>9}")
cp, cd = Counter(tuple(e[1]) for e in pre), Counter(tuple(e[1]) for e in dec)
for op in sorted(set(cp) | set(cd)):
    flag = "" if cp.get(op, 0) == cd.get(op, 0) else "   <-- differs"
    print(f"{' '.join(op):<28}{cp.get(op,0):>9}{cd.get(op,0):>9}{flag}")

## 7 — Question 2: were the inferred shapes right?

Side by side: what `dynshape`'s hand-written query/key split **predicted**, against what the hardware
trace **shows**. This is the moment the guess is graded.

In [ ]:
DYN = "/content/dynamic_shape_power_sim"
if not os.path.isdir(DYN):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/shubhamOjha1000/dynamic_shape_power_sim.git",
                    DYN], check=True)
sys.path.insert(0, DYN)

from dynshape import ShapeRewriter

rw = ShapeRewriter.from_dir(os.path.join(DYN, "templates", "gpt2"))
print("decode_source:", rw.decode_source, "(no decode traces installed yet)")

B, S = 8, 128
inferred = norm(rw.expand(B, S, "decode"))
measured = decode_traces[(B, S)]

if len(inferred) != len(measured):
    print(f"\nCannot compare entry by entry: {len(inferred)} inferred vs "
          f"{len(measured)} measured. That difference IS the finding.")
else:
    diffs = [(i, a, b) for i, (a, b) in enumerate(zip(inferred, measured)) if a != b]
    print(f"\n{len(measured) - len(diffs)} / {len(measured)} entries match exactly")
    if not diffs:
        print("\nThe inferred query/key split was CORRECT. It is now measured, not argued.")
    else:
        print(f"\n{len(diffs)} entries differ. First few:\n")
        for i, a, b in diffs[:6]:
            print(f"  entry {i}  ({' '.join(a[1])})")
            print(f"    inferred: {a[0]}")
            print(f"    measured: {b[0]}")
            print()

## 8 — Question 3: does a learned decode law generalise?

In [ ]:
from dynshape import learn_scaling, rewrite_dims

base    = decode_traces[(8, 128)]
alt_b   = decode_traces[(16, 128)]
alt_s   = decode_traces[(8, 512)]
heldout = decode_traces[(16, 512)]

try:
    rules = learn_scaling(base, alt_b, alt_s, batch_ratio=2, seq_ratio=4)
    print(f"learned a decode law over {sum(len(r) for r in rules)} numeric fields\n")

    generated = rewrite_dims(base, rules, 8, 128, 16, 512)
    if generated == heldout:
        print("HELD-OUT TEST PASSED -- b16 s512 derived from the three anchors is")
        print("identical to its independent trace. Decode is now as verified as prefill.")
    else:
        bad = [i for i, (a, b) in enumerate(zip(generated, heldout)) if a != b]
        print(f"HELD-OUT TEST FAILED at {len(bad)} of {len(heldout)} entries. First:")
        i = bad[0]
        print(f"  derived: {generated[i][0]}")
        print(f"  traced : {heldout[i][0]}")
        print("\nA pure power law does not describe decode. Worth knowing --")
        print("it means decode needs interpolation over traces, not extrapolation from three.")
except ValueError as e:
    print("learn_scaling refused:", e)
    print("\nThat is the loud failure working as designed: some field does not")
    print("follow value = const * B^a * S^b, so no law was silently invented.")

## 9 — Install the traces and download them

Copy the three anchors into `templates/gpt2/` and `dynshape` picks them up automatically —
`decode_source` flips from `inferred` to `measured`, and the hand-written split rule is never
consulted again.

The held-out `b16 s512` is deliberately **not** installed: it stays a test, not training data.

In [ ]:
import shutil, glob

TEMPLATES = os.path.join(DYN, "templates", "gpt2")
for b, s in [(8, 128), (16, 128), (8, 512)]:
    shutil.copy(os.path.join(OUT, f"gpt2model_gpt2_pbf16_b{b}_s{s}_modedecode.json"), TEMPLATES)

rw2 = ShapeRewriter.from_dir(TEMPLATES)
print("decode_source:", rw2.decode_source)
print("prefill kernels:", rw2.n_kernels("prefill"))
print("decode  kernels:", rw2.n_kernels("decode"))

shutil.make_archive("/content/gpt2_decode_traces", "zip", OUT)
print("\nzipped:", sorted(os.path.basename(f) for f in glob.glob(OUT + "/*.json")))

try:
    from google.colab import files
    files.download("/content/gpt2_decode_traces.zip")
except Exception as e:
    print("(not in Colab -- grab /content/gpt2_decode_traces.zip manually)", e)

---

## What to do with the result

Commit the three `..._modedecode.json` files into `templates/gpt2/` in the
[dynamic_shape_power_sim](https://github.com/shubhamOjha1000/dynamic_shape_power_sim) repo. Nothing
else changes — `from_dir` finds them, and every decode number afterwards comes from a measured law.

### Read the outcomes honestly

| section | outcome | meaning |
|---|---|---|
| **4 control** | mismatch | stop. Nothing below is interpretable |
| **6** | op sequence identical | the structural assumption held |
| **6** | op sequence differs | the inferred rule was wrong about structure — the most valuable result here |
| **7** | all entries match | the guess was right, and is now a measurement |
| **7** | entries differ | every decode power number produced so far was wrong; these traces are the fix |
| **8** | held-out passes | decode is as verified as prefill |
| **8** | held-out fails | decode is not a pure power law; it needs interpolation across more traces |

### What these traces still cannot fix

The **KV-cache write** stays missing. HuggingFace appends with `torch.cat`, and `cat` is absent from
the tracer's keep-list (`linear, addmm, matmul, bmm, mm, mul, add, ... softmax, layernorm`), so it
produces no entry in any mode.

That is fine for a vLLM-targeted simulator and the size is known: under paged attention the write is
≈295 KB per step for GPT-2 at batch 8, ctx 2048 — about **0.014%** of a decode step. Under
HuggingFace's `torch.cat` it would be ≈1.2 GB per step, **~56%** — which is why HuggingFace
measurements are not a valid comparison target for decode timing.